# DINOv2 MLP Experiments

Semi-supervised anomaly detection using DINOv2 features and a patch-level MLP classifier with Hard Negative Mining (HNM) and Test-Time Augmentation (TTA).

In [1]:
from pathlib import Path
import csv
import importlib
import os
import sys

import numpy as np
import pandas as pd
import yaml
try:
    from IPython.display import display
except ImportError:
    display = print

ROOT = Path.cwd().resolve()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))

from src.common.config import load_config
from src.common.data import SpacepressoDataModule
from src.common.evaluation import evaluate_predictions
from src.common.paths import resolve_path
from src.common.prediction_processing import process_prediction_maps
from src.common.q8rle import float_matrix_to_q8rle
from src.common.seed import set_seed
from src.common.submission import _prepare_prediction_map, validate_submission
import src.common.tuning as tuning_module
tuning_module = importlib.reload(tuning_module)
OptunaTuner = tuning_module.OptunaTuner
suggest_efficientad_dinov2_config = tuning_module.suggest_efficientad_dinov2_config
from src.common.training import ExperimentRunner
from src.common.validation import make_validation_split
from src.common.visualization import show_predictions
from src.methods import get_method_class

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from PIL import Image
from tqdm.auto import tqdm
from scipy.ndimage import gaussian_filter
from src.methods.base import BaseMethod

class _MLP(nn.Module):
    def __init__(self, in_dim: int, hidden: int = 256, dropout: float = 0.2):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden, hidden), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden, 1),
        )
    def forward(self, x): return self.net(x).squeeze(-1)

def _focal_loss(logits, targets, alpha, gamma):
    bce = F.binary_cross_entropy_with_logits(logits, targets, reduction="none")
    pt = torch.exp(-bce)
    at = targets * alpha + (1.0 - targets) * (1.0 - alpha)
    return (at * (1.0 - pt) ** gamma * bce).mean()

class SupervisedDinov2MLP(BaseMethod):
    def __init__(self, config):
        super().__init__(config)
        mc = config.get("method", {})
        self.device = torch.device(mc.get("device", "cuda" if torch.cuda.is_available() else "cpu"))
        self.model_name = mc.get("dinov2_model", "dinov2_vits14")
        print(f"Loading DINOv2 backbone: {self.model_name}")
        self.model = torch.hub.load("facebookresearch/dinov2", self.model_name).to(self.device).eval()
        self.image_size = config.get("data", {}).get("image_size", 224)
        if isinstance(self.image_size, int): self.image_size = (self.image_size, self.image_size)
        
        # multi-layer settings
        self.n_layers = mc.get("n_layers", 4)
        
        self.mlp_hidden, self.mlp_epochs = mc.get("mlp_hidden", 256), mc.get("mlp_epochs", 20)
        self.mlp_lr, self.sigma = mc.get("mlp_lr", 1e-3), mc.get("sigma", 4.0)
        self.focal_alpha, self.focal_gamma = mc.get("focal_alpha", None), mc.get("focal_gamma", 2.0)
        self.class_wise, self.seed = mc.get("class_wise", True), config.get("seed", 42)
        self.fg_threshold = mc.get("fg_threshold", 0.1) # Intensity threshold for foreground
        
        self.mlps, self.feat_norms = {}, {}
        with torch.no_grad():
            dummy = torch.zeros(1, 3, self.image_size[0], self.image_size[1]).to(self.device)
            tokens = self._extract_tokens(dummy)
            self.feat_dim, self.grid_h = tokens.shape[-1], int(tokens.shape[1]**0.5)
            self.grid_w = self.grid_h
            print(f"DINOv2 (Multi-Scale) Grid: {self.grid_h}x{self.grid_w} | Multi-Layer Dim: {self.feat_dim}")

    def _extract_tokens(self, images):
        mean = torch.tensor([0.485, 0.456, 0.406]).view(1,3,1,1).to(self.device)
        std  = torch.tensor([0.229, 0.224, 0.225]).view(1,3,1,1).to(self.device)
        x = (images.to(self.device) - mean) / std
        
        # Extract and concatenate multiple layers
        if hasattr(self.model, "get_intermediate_layers"):
            layers = self.model.get_intermediate_layers(x, n=self.n_layers)
            tokens = torch.cat(layers, dim=-1)
        else:
            tokens = self.model.forward_features(x)["x_norm_patchtokens"]
            
        return F.normalize(tokens.float(), dim=-1)

    def fit(self, train_data, val_data=None):
        grouped = self._group(list(train_data)) if self.class_wise else {"__global__": list(train_data)}
        for key, samples in sorted(grouped.items()):
            print(f"Fitting Enhanced DINOv2-MLP for {key}: {len(samples)} images")
            feats, labels, f_mean, f_std = self._build_patch_dataset(samples)
            self.feat_norms[key] = (f_mean, f_std)
            feats = (feats - f_mean) / f_std
            print(f"  Pass 1: Training initial MLP...")
            mlp = self._train_mlp(feats, labels, self.mlp_epochs)
            print("  Pass 2: Mining hard negatives and retraining...")
            with torch.no_grad():
                neg_idx = torch.where(labels == 0)[0]
                scores = torch.cat([torch.sigmoid(mlp(feats[neg_idx][i:i+4096].to(self.device))).cpu() for i in range(0, len(neg_idx), 4096)])
                _, hard_rel = torch.topk(scores, min(int(labels.sum().item()) * 3, len(scores)))
                keep = torch.cat([torch.where(labels == 1)[0], neg_idx[hard_rel]])
            self.mlps[key] = self._train_mlp(feats[keep], labels[keep], self.mlp_epochs // 2, init_state=mlp.state_dict())
        return self

    def _train_mlp(self, feats, labels, epochs, init_state=None):
        mlp = _MLP(self.feat_dim, self.mlp_hidden).to(self.device)
        if init_state: mlp.load_state_dict(init_state)
        opt = torch.optim.Adam(mlp.parameters(), lr=self.mlp_lr)
        alpha = float(self.focal_alpha) if self.focal_alpha is not None else min(0.95, 1.0 - labels.sum().item() / len(labels))
        loader = torch.utils.data.DataLoader(torch.utils.data.TensorDataset(feats, labels), batch_size=4096, shuffle=True)
        for e in range(1, epochs + 1):
            mlp.train()
            for xb, yb in loader:
                xb, yb = xb.to(self.device), yb.to(self.device)
                opt.zero_grad()
                loss = _focal_loss(mlp(xb).squeeze(-1), yb, alpha, self.focal_gamma)
                loss.backward(); opt.step()
        return mlp.eval()

    def _build_patch_dataset(self, samples):
        all_feats, all_labels, gen = [], [], torch.Generator(); gen.manual_seed(self.seed)
        for s in tqdm(samples, desc="Extracting DINOv2 tokens", leave=False):
            img = s.image if s.image is not None else self._load_image(s.image_path)
            with torch.no_grad(): tokens = self._extract_tokens(torch.from_numpy(img).permute(2,0,1).unsqueeze(0).to(self.device)).view(-1, self.feat_dim).cpu()
            if s.label == 1 and s.mask_path:
                mask = np.asarray(Image.open(s.mask_path).convert("L").resize((self.grid_w, self.grid_h), resample=Image.NEAREST)) > 0
                all_feats.append(tokens); all_labels.append(torch.from_numpy(mask.flatten().astype(np.float32)))
            else:
                idx = torch.randperm(tokens.shape[0], generator=gen)[:min(50, tokens.shape[0])]
                all_feats.append(tokens[idx]); all_labels.append(torch.zeros(len(idx)))
        feats, labels = torch.cat(all_feats), torch.cat(all_labels)
        norm_f = feats[labels == 0]
        return feats, labels, norm_f.mean(0), norm_f.std(0).clamp(min=1e-6)

    def predict(self, test_data, tta=False):
        predictions = {}
        grouped = self._group(list(test_data)) if self.class_wise else {"__global__": list(test_data)}
        for key, samples in grouped.items():
            mlp_key = key if key in self.mlps else "__global__"
            mlp, (f_mean, f_std) = self.mlps[mlp_key], self.feat_norms[mlp_key]
            f_mean, f_std = f_mean.to(self.device), f_std.to(self.device)
            for s in tqdm(samples, desc=f"Inference {key}", leave=False):
                img = self._load_image(s.image_path)
                img_t = torch.from_numpy(img).permute(2,0,1).float()
                
                gray = np.mean(img, axis=-1)
                fg_mask = (gray > self.fg_threshold).astype(np.float32)
                
                if tta:
                    k_vals = [0, 1, 2, 3] if s.class_name in ["class_03", "class_05", "class_06", "class_07"] else [0, 2]
                    aug_t = []
                    for k, f in [(k, f) for k in k_vals for f in [False, True]]:
                        a = torch.rot90(img_t, k, [1, 2])
                        if f: a = torch.flip(a, [2])
                        aug_t.append(a)
                    with torch.no_grad():
                        scores = torch.sigmoid(mlp((self._extract_tokens(torch.stack(aug_t).to(self.device)) - f_mean) / f_std).squeeze(-1)).view(len(aug_t), self.grid_h, self.grid_w).cpu().numpy()
                    aug_preds = []
                    for i, (k, f) in enumerate([(k, f) for k in k_vals for f in [False, True]]):
                        p = scores[i]
                        if f: p = np.fliplr(p)
                        aug_preds.append(np.rot90(p, -k % 4))
                    final_grid = np.mean(aug_preds, axis=0)
                else:
                    with torch.no_grad(): final_grid = torch.sigmoid(mlp((self._extract_tokens(img_t.unsqueeze(0).to(self.device)) - f_mean) / f_std).squeeze(-1)).view(self.grid_h, self.grid_w).cpu().numpy()
                
                amap = np.asarray(Image.fromarray(final_grid).resize(self.image_size[::-1], resample=Image.BILINEAR))
                amap = amap * fg_mask 
                if self.sigma > 0: amap = gaussian_filter(amap, sigma=self.sigma)
                predictions[str(s.image_id)] = amap.astype(np.float16)
        return predictions

    def _load_image(self, path):
        with Image.open(path) as im: return np.asarray(im.convert("RGB").resize(self.image_size[::-1], resample=Image.BILINEAR), dtype=np.float32) / 255.0

    def _group(self, samples):
        g = {}
        for s in samples: g.setdefault(s.class_name, []).append(s)
        return g

print("Enhanced SupervisedDinov2MLP (Multi-Scale + Masking) implemented.")

Enhanced SupervisedDinov2MLP (Multi-Scale + Masking) implemented.


In [ ]:
SEED = 42
set_seed(SEED)

MLP_CONFIG = {
    "experiment": {"name": "supervised_dinov2_mlp_v2_juan"},
    "method": {
        "name": "supervised_dinov2_mlp",
        "dinov2_model": "dinov2_vits14",
        "n_layers": 4,
        "fg_threshold": 0.1,
        "mlp_hidden": 512,
        "mlp_epochs": 20,
        "mlp_lr": 1e-3,
        "sigma": 4.0,
        "tta_aggregation": "mean",
        "class_wise": True
    },
    "data": {"image_size": 224},
    "seed": SEED
}

WRITE_SUBMISSION = True
SUBMISSION_CHUNK_SIZE = 32
SUBMISSION_TEST_LIMIT = None 

In [ ]:
# Load Data
config = load_config(ROOT / "configs/efficientad_dinov2/juan_tuning.yaml")
data_root_candidates = [
    ROOT / "data" / "spacepresso" / "adl-2025-2026-anomaly-detection",
    ROOT / "data" / "spacepresso",
]
resolved_data_root = next((path for path in data_root_candidates if path.exists()), None)
if resolved_data_root is not None:
    config["data"]["root"] = str(resolved_data_root)
config["data"]["load_images"] = False # We load images on the fly

dm = SpacepressoDataModule(**config["data"])
train_good_all = dm.load_train_good()
train_anomalies_all = dm.load_train_anomalies()

print({
    "train_good": len(train_good_all),
    "train_anomalies": len(train_anomalies_all),
})

{'train_good': 19005, 'train_anomalies': 235}


In [ ]:
print("Initializing SupervisedDinov2MLP...")
model = SupervisedDinov2MLP(MLP_CONFIG)
print("Training on combined normal + anomalous data (HNM enabled)...")
model.fit(train_good_all + train_anomalies_all)

Initializing SupervisedDinov2MLP...
Loading DINOv2 backbone: dinov2_vits14


Using cache found in C:\Users\sjuan/.cache\torch\hub\facebookresearch_dinov2_main
C:\Users\sjuan/.cache\torch\hub\facebookresearch_dinov2_main\dinov2\layers\swiglu_ffn.py:51: UserWarning: xFormers is not available (SwiGLU)
  warnings.warn("xFormers is not available (SwiGLU)")
C:\Users\sjuan/.cache\torch\hub\facebookresearch_dinov2_main\dinov2\layers\attention.py:33: UserWarning: xFormers is not available (Attention)
  warnings.warn("xFormers is not available (Attention)")
C:\Users\sjuan/.cache\torch\hub\facebookresearch_dinov2_main\dinov2\layers\block.py:40: UserWarning: xFormers is not available (Block)
  warnings.warn("xFormers is not available (Block)")


DINOv2 (Multi-Scale) Grid: 16x16 | Multi-Layer Dim: 1536
Training on combined normal + anomalous data (HNM enabled)...
Fitting Enhanced DINOv2-MLP for class_01: 2625 images


Extracting DINOv2 tokens:   0%|          | 0/2625 [00:00<?, ?it/s]

  Pass 1: Training initial MLP...
  Pass 2: Mining hard negatives and retraining...
Fitting Enhanced DINOv2-MLP for class_02: 2165 images


...

  Pass 1: Training initial MLP...
  Pass 2: Mining hard negatives and retraining...
Fitting Enhanced DINOv2-MLP for class_07: 2180 images


Extracting DINOv2 tokens:   0%|          | 0/2180 [00:00<?, ?it/s]

  Pass 1: Training initial MLP...
  Pass 2: Mining hard negatives and retraining...
Fitting Enhanced DINOv2-MLP for class_08: 2080 images


Extracting DINOv2 tokens:   0%|          | 0/2080 [00:00<?, ?it/s]

  Pass 1: Training initial MLP...
  Pass 2: Mining hard negatives and retraining...


In [ ]:
if WRITE_SUBMISSION:
    test_samples = sorted(dm.load_test(), key=lambda sample: sample.image_id)
    if SUBMISSION_TEST_LIMIT is not None:
        test_samples = test_samples[: int(SUBMISSION_TEST_LIMIT)]
    
    stamp = pd.Timestamp.now().strftime("%Y%m%d_%H%M%S")
    output_path = resolve_path(f"submissions/juan/supervised_dinov2_mlp_{stamp}.csv", ROOT)
    output_path.parent.mkdir(parents=True, exist_ok=True)

    print(f"Generating submission for {len(test_samples)} images...")
    written = 0
    with output_path.open("w", newline="", encoding="utf-8") as handle:
        writer = csv.DictWriter(handle, fieldnames=["ID", "Label"])
        writer.writeheader()
        
        for start in range(0, len(test_samples), int(SUBMISSION_CHUNK_SIZE)):
            chunk = test_samples[start : start + int(SUBMISSION_CHUNK_SIZE)]
            chunk_predictions = model.predict(chunk, tta=True)
            
            for sample in chunk:
                anomaly_map = _prepare_prediction_map(chunk_predictions[sample.image_id])
                writer.writerow({"ID": sample.image_id, "Label": float_matrix_to_q8rle(anomaly_map)})
                written += 1
            del chunk_predictions

    validate_submission(output_path, expected_shape=(224, 224))
    print({"submission_path": str(output_path), "rows": written})
else:
    print("WRITE_SUBMISSION is False.")

Generating submission for 5910 images...


Inference class_03:   0%|          | 0/5 [00:00<?, ?it/s]

Inference class_05:   0%|          | 0/5 [00:00<?, ?it/s]

Inference class_08:   0%|          | 0/5 [00:00<?, ?it/s]

Inference class_02:   0%|          | 0/10 [00:00<?, ?it/s]

...

Inference class_07:   0%|          | 0/3 [00:00<?, ?it/s]

Inference class_07:   0%|          | 0/2 [00:00<?, ?it/s]

Inference class_08:   0%|          | 0/15 [00:00<?, ?it/s]

Inference class_06:   0%|          | 0/5 [00:00<?, ?it/s]

{'submission_path': 'C:\\Users\\sjuan\\Desktop\\Spacespresso\\submissions\\juan\\supervised_dinov2_mlp_20260514_200602.csv', 'rows': 5910}
